In [7]:
# nạp các hàm xử lý dữ liệu và đánh giá.

import os
import numpy as np
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support
from IPython.display import display

def find_column(df, names):
    for n in df.columns:
        if n.lower() in [x.lower() for x in names]:
            return n
    return None

def generate_ratings_for_sheet(df, as_float=False, seed=42):
    np.random.seed(seed)
    working = df.copy()
    
    # Đảm bảo có các cột ID
    user_col = find_column(working, ['UserID', 'UserId', 'user_id', 'user'])
    course_col = find_column(working, ['CourseID', 'CourseId', 'course_id', 'course'])
    if user_col is None or course_col is None:
        raise ValueError('Không tìm thấy cột UserID hoặc CourseID trong dữ liệu')

    # Tính toán độ lệch (bias) dựa trên mức độ phổ biến của khóa học
    course_counts = working[course_col].value_counts().astype(float)
    cc_min, cc_max = course_counts.min(), course_counts.max()
    if cc_max - cc_min <= 0:
        norm_counts = course_counts * 0.0
    else:
        norm_counts = (course_counts - cc_min) / (cc_max - cc_min)

    working['_course_pop'] = working[course_col].map(norm_counts).fillna(0.0)

    # Tính toán độ lệch dựa trên mức độ hoạt động của user
    user_counts = working[user_col].value_counts().astype(float)
    uc_min, uc_max = user_counts.min(), user_counts.max()
    if uc_max - uc_min <= 0:
        norm_users = user_counts * 0.0
    else:
        norm_users = (user_counts - uc_min) / (uc_max - uc_min)
    working['_user_act'] = working[user_col].map(norm_users).fillna(0.0)

    # Base mean rating và effect sizes
    base = 3.2
    course_effect = working['_course_pop'] * 1.2  # Tăng tối đa +1.2
    user_effect = (working['_user_act'] - 0.5) * 0.6  # Nằm trong khoảng [-0.3, +0.3]

    raw_means = base + course_effect + user_effect

    # Thêm nhiễu ngẫu nhiên và giới hạn trong khoảng 1..5
    noise = np.random.normal(0, 0.6, size=len(working))
    raw = raw_means + noise
    
    ratings = np.clip(np.round(raw, 2) if as_float else np.rint(raw), 1, 5)

    if as_float:
        working['Rating'] = ratings.astype(float)
    else:
        working['Rating'] = ratings.astype(int)

    # Xóa các cột phụ trợ
    working = working.drop(columns=['_course_pop', '_user_act'])
    return working

def evaluate_with_noisy_prediction(df, rating_col='Rating', noise_std=1.0, threshold=4.0, seed=42):
    np.random.seed(seed + 1)
    if rating_col not in df.columns:
        raise ValueError('Không tìm thấy cột rating_col trong dataframe')
        
    true = df[rating_col].astype(float).to_numpy()
    pred = np.round(np.clip(true + np.random.normal(0, noise_std, size=len(true)), 1.0, 5.0), 2)
    
    true_rel = (true >= threshold).astype(int)
    pred_rel = (pred >= threshold).astype(int)
    
    p, r, f1, _ = precision_recall_fscore_support(true_rel, pred_rel, average='binary', zero_division=0)
    return p, r, f1

In [10]:
# Cấu hình biến và thực thi

# Cấu hình đường dẫn và tham số (thay thế cho argparse)
input_interactions = r'D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions.csv'
output_interactions = r'D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions_with_ratings.csv'

as_float = False
noise_std = 1.0
threshold = 5.0
force = True # Đặt True để ghi đè nếu cột Rating đã tồn tại
seed = 42

if not os.path.exists(input_interactions):
    print(f'Không tìm thấy file: {input_interactions}')
else:
    # 1. Đọc dữ liệu
    df_interactions = pd.read_csv(input_interactions)
    print(f"Đã tải {len(df_interactions)} dòng dữ liệu tương tác.")

    # 2. Tạo hoặc điền Rating
    if 'Rating' in df_interactions.columns and not force:
        print('Cột Rating đã tồn tại. Chuyển force=True để ghi đè.')
    else:
        df_interactions = generate_ratings_for_sheet(df_interactions, as_float=as_float, seed=seed)
        print(f"Đã điền Rating cho {len(df_interactions)} dòng.")

    # 3. Đánh giá mô phỏng
    p, r, f1 = evaluate_with_noisy_prediction(
        df_interactions, 
        rating_col='Rating', 
        noise_std=noise_std, 
        threshold=threshold, 
        seed=seed
    )
    
    print(f'\n--- Kết quả đánh giá (simulated predictions) với threshold {threshold} ---')
    print(f'Precision: {p:.4f}')
    print(f'Recall:    {r:.4f}')
    print(f'F1-score:  {f1:.4f}\n')

    # 4. Lưu lại thành file CSV mới
    df_interactions.to_csv(output_interactions, index=False)
    print(f'Đã lưu file chứa mock ratings tại: {output_interactions}')
    
    # 5. Hiển thị trực tiếp dưới cell
    display(df_interactions.head(10))

Đã tải 184065 dòng dữ liệu tương tác.
Đã điền Rating cho 184065 dòng.

--- Kết quả đánh giá (simulated predictions) với threshold 5.0 ---
Precision: 0.2157
Recall:    0.5075
F1-score:  0.3027

Đã lưu file chứa mock ratings tại: D:\KLKS\Neo4j\online-course-recommendation-system\mock_user_interactions_with_ratings.csv


,UserID,CourseID,Rating
0,3957,583,3
1,928,444,3
2,6935,660,3
3,6582,637,4
4,4460,1131,3
5,7209,107,3
6,716,619,4
7,9948,587,4
8,758,131,3
9,3656,5,4
